# NeuroScan: train on Colab

Clones the repository at a chosen branch, tag or commit, trains one or more configs and writes a
**run bundle** per config to Google Drive. No MLflow here: bundles are imported into MLflow on the
tracking machine afterwards (`make import-all`).

**Before the first run**
1. In Colab, open 🔑 **Secrets** (left sidebar), add `KAGGLE_API_TOKEN` with your Kaggle token, and enable
   notebook access.
2. **Runtime → Change runtime type → GPU.**

Every bundle records the exact commit it was trained from. If the session disconnects, run all cells
again with the same settings: finished configs are skipped and the interrupted one resumes from its
last finished epoch.

In [ ]:
# Settings
REPO = "https://github.com/AminRezaeeyan/NeuroScan.git"
REF = "main"  # branch, tag or commit to train from
CONFIGS = [
    "configs/efficientnet_b0_shortcut_reduced.yaml",
    # "configs/baseline_cnn.yaml", "configs/cnn_gap.yaml",
    # "configs/efficientnet_b0.yaml", "configs/resnet50.yaml",
]
DRY_RUN = False  # True: 1 epoch per stage on 32 images, to test the pipeline

DRIVE_ROOT = "/content/drive/MyDrive/neuroscan"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: change the runtime type"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Get the code

In [ ]:
!rm -rf /content/NeuroScan
!git clone --quiet --filter=blob:none {REPO} /content/NeuroScan
%cd /content/NeuroScan
!git checkout --quiet {REF}
!git log -1 --format="Training from %H (%cd)" --date=short
!pip install -q -e .
!python -c "import tensorflow as tf, keras, neuroscan; print('TF', tf.__version__, '| Keras', keras.__version__, '| neuroscan', neuroscan.__version__)"

## Download the dataset

In [ ]:
import os

from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
!python -m neuroscan.data.download --out /content/data/raw

## Train

In [ ]:
# TF_CPP_MIN_LOG_LEVEL=2 hides TensorFlow's C++ info/warning logs (e.g. cuDNN autotuning
# skipping "potentially buggy" algorithms), which are harmless and flood the output.
# A config whose bundle already exists is reported as finished and skipped.
flags = "--dry-run" if DRY_RUN else ""
for config in CONFIGS:
    print(f"=== {config}")
    !TF_CPP_MIN_LOG_LEVEL=2 python -m neuroscan.train --config {config} --raw-dir /content/data/raw --out {DRIVE_ROOT}/runs {flags}

In [ ]:
!ls -la {DRIVE_ROOT}/runs/

## Next: on the tracking machine

1. Download the new folders from `MyDrive/neuroscan/runs/` into `runs/` in the repository.
2. `make import-all` imports every bundle not yet in MLflow and prints the run IDs.
3. `make eval RUN=<run-id> SETS="val test"` for each new run, then `make select`.